# Robust Estimation

Handle outliers with robust standard errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.models import fit_glm

# Generate data with an outlier
np.random.seed(42)
n = 100
x = np.random.randn(n)
y = 2 * x + 1 + np.random.randn(n) * 0.5  # True: y = 1 + 2*x + noise

# Add extreme outlier
y[0] = 50  
outlier_idx = 0

# Fit standard GLM (let fit_glm handle intercept)
result = fit_glm(x.reshape(-1, 1), y, family='gaussian', fit_intercept=True)

# Get coefficients (including intercept)
beta = np.array([result.intercept_, result.coef_[0]])
se = np.array([result.intercept_std_error_, result.std_errors_[0]])

print("="*60)
print("STANDARD OLS ESTIMATION")
print("="*60)
print(f"Coefficients: β₀={beta[0]:.3f}, β₁={beta[1]:.3f}")
print(f"Standard Errors: SE(β₀)={se[0]:.3f}, SE(β₁)={se[1]:.3f}")
print(f"True parameters: β₀=1.000, β₁=2.000")
print()

# Create full design matrix for later calculations (with intercept)
X = np.column_stack([np.ones(n), x])

In [ ]:
# Compute Huber-White Robust (Sandwich) Standard Errors
# Using the built-in function from Aurora-GLM
from aurora.inference import robust_covariance

# Get robust standard errors using HC0 (basic White estimator)
robust_result = robust_covariance(result, hc_type='HC0')

# Get residuals for visualization
y_pred = result.predict(x.reshape(-1, 1))
resid = y - y_pred

print("="*60)
print("ROBUST (SANDWICH/HUBER-WHITE) STANDARD ERRORS")
print("="*60)
print(f"Standard SE:     SE(β₀)={se[0]:.3f}, SE(β₁)={se[1]:.3f}")
print(f"Robust SE (HC0): SE(β₀)={robust_result.intercept_std_error:.3f}, SE(β₁)={robust_result.std_errors[0]:.3f}")
print(f"\nRatio (Robust/Standard):")
print(f"  β₀: {robust_result.intercept_std_error/se[0]:.2f}x")
print(f"  β₁: {robust_result.std_errors[0]/se[1]:.2f}x")
print("\nInterpretation: Robust SEs are much larger due to the outlier,")
print("indicating increased uncertainty in the parameter estimates.")

In [ ]:
# Visualize the effect of the outlier
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Data with outlier
ax1 = axes[0]
ax1.scatter(x[1:], y[1:], alpha=0.6, label='Normal data')
ax1.scatter(x[outlier_idx], y[outlier_idx], color='red', s=200, marker='*', 
            label='Outlier', zorder=5)
x_line = np.linspace(x.min(), x.max(), 100)
y_pred_line = beta[0] + beta[1] * x_line
ax1.plot(x_line, y_pred_line, 'b-', linewidth=2, label='Fitted line (with outlier)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Data with Outlier')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Residuals
ax2 = axes[1]
ax2.scatter(y_pred[1:], resid[1:], alpha=0.6, label='Normal residuals')
ax2.scatter(y_pred[outlier_idx], resid[outlier_idx], color='red', 
            s=200, marker='*', label='Outlier residual', zorder=5)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('Fitted values')
ax2.set_ylabel('Residuals')
ax2.set_title('Residual Plot')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Compare with and without outlier
ax3 = axes[2]
# Fit without outlier (use x without intercept column)
x_no_outlier = x[1:]
y_no_outlier = y[1:]
result_no_outlier = fit_glm(x_no_outlier.reshape(-1, 1), y_no_outlier, family='gaussian', fit_intercept=True)
beta_no = np.array([result_no_outlier.intercept_, result_no_outlier.coef_[0]])

ax3.scatter(x[1:], y[1:], alpha=0.6, label='Normal data')
ax3.scatter(x[outlier_idx], y[outlier_idx], color='red', s=200, marker='*', 
            label='Outlier (excluded)', zorder=5)
y_pred_no_outlier = beta_no[0] + beta_no[1] * x_line
ax3.plot(x_line, y_pred_line, 'b-', linewidth=2, label='With outlier', alpha=0.5)
ax3.plot(x_line, y_pred_no_outlier, 'g-', linewidth=2, label='Without outlier')
ax3.plot(x_line, 1 + 2*x_line, 'k--', linewidth=1, label='True line', alpha=0.5)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Effect of Outlier on Fit')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCOMPARISON: WITH vs WITHOUT OUTLIER")
print("="*60)
print(f"With outlier:    β₀={beta[0]:.3f}, β₁={beta[1]:.3f}")
print(f"Without outlier: β₀={beta_no[0]:.3f}, β₁={beta_no[1]:.3f}")
print(f"True values:     β₀=1.000, β₁=2.000")

## Heteroscedasticity-Consistent (HC) Standard Error Variants

There are several variants of robust standard errors (HC0, HC1, HC2, HC3, HC4).
Let's implement the most common ones:

In [ ]:
# Compare different HC variants using the built-in function
from aurora.inference import robust_covariance

print("="*70)
print("COMPARISON OF DIFFERENT HC STANDARD ERROR ESTIMATORS")
print("="*70)
print(f"{'Method':<10} {'SE(β₀)':<12} {'SE(β₁)':<12} {'Description':<30}")
print("-"*70)

print(f"{'OLS':<10} {se[0]:<12.4f} {se[1]:<12.4f} {'Standard (assumes homoscedasticity)':<30}")

hc_types = ['HC0', 'HC1', 'HC2', 'HC3', 'HC4']
descriptions = [
    'White (1980) - basic',
    'DF correction',
    'Leverage correction',
    'Better for small samples',
    'Best for influential points'
]

for hc_type, desc in zip(hc_types, descriptions):
    robust_res = robust_covariance(result, hc_type=hc_type)
    print(f"{hc_type:<10} {robust_res.intercept_std_error:<12.4f} {robust_res.std_errors[0]:<12.4f} {desc:<30}")

print("\nRecommendation: HC3 is generally preferred for small to moderate samples.")
print("\nNote: All HC methods are now available via aurora.inference.robust_covariance()")

## Bootstrap Standard Errors

Another robust approach is to use bootstrap resampling to estimate standard errors:

In [ ]:
# Bootstrap inference using the built-in function
from aurora.inference import bootstrap_inference

print("Computing bootstrap estimates (this may take a moment)...")
boot_result = bootstrap_inference(result, n_bootstrap=1000, seed=42)

# Also get HC3 for comparison
hc3_result = robust_covariance(result, hc_type='HC3')

print("\n" + "="*70)
print("BOOTSTRAP STANDARD ERRORS")
print("="*70)
print(f"                β₀              β₁")
print("-"*70)
print(f"Estimate:       {beta[0]:6.3f}          {beta[1]:6.3f}")
print(f"OLS SE:         {se[0]:6.3f}          {se[1]:6.3f}")
print(f"HC3 SE:         {hc3_result.intercept_std_error:6.3f}          {hc3_result.std_errors[0]:6.3f}")
print(f"Bootstrap SE:   {boot_result['intercept_std_error']:6.3f}          {boot_result['std_errors'][0]:6.3f}")
print(f"\nBootstrap 95% CI:")
print(f"β₀: [{boot_result['intercept_ci'][0]:6.3f}, {boot_result['intercept_ci'][1]:6.3f}]")
print(f"β₁: [{boot_result['ci_lower'][0]:6.3f}, {boot_result['ci_upper'][0]:6.3f}]")
print("\nNote: Bootstrap inference is now available via aurora.inference.bootstrap_inference()")

In [ ]:
# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

boot_coefs = boot_result['boot_coefs']

# β₀ distribution
ax1 = axes[0]
ax1.hist(boot_coefs[:, 0], bins=50, density=True, alpha=0.7, edgecolor='black')
ax1.axvline(beta[0], color='red', linestyle='--', linewidth=2, label='Estimate')
ax1.axvline(1.0, color='green', linestyle='--', linewidth=2, label='True value')
ax1.axvline(boot_result['intercept_ci'][0], color='blue', linestyle=':', linewidth=2, label='95% CI')
ax1.axvline(boot_result['intercept_ci'][1], color='blue', linestyle=':', linewidth=2)
ax1.set_xlabel('β₀ (Intercept)')
ax1.set_ylabel('Density')
ax1.set_title('Bootstrap Distribution of β₀')
ax1.legend()
ax1.grid(True, alpha=0.3)

# β₁ distribution
ax2 = axes[1]
ax2.hist(boot_coefs[:, 1], bins=50, density=True, alpha=0.7, edgecolor='black')
ax2.axvline(beta[1], color='red', linestyle='--', linewidth=2, label='Estimate')
ax2.axvline(2.0, color='green', linestyle='--', linewidth=2, label='True value')
ax2.axvline(boot_result['ci_lower'][0], color='blue', linestyle=':', linewidth=2, label='95% CI')
ax2.axvline(boot_result['ci_upper'][0], color='blue', linestyle=':', linewidth=2)
ax2.set_xlabel('β₁ (Slope)')
ax2.set_ylabel('Density')
ax2.set_title('Bootstrap Distribution of β₁')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary: When to Use Robust Standard Errors

**Use robust standard errors when:**
- You suspect heteroscedasticity (non-constant variance)
- There are potential outliers or influential points
- You want inference that's valid under misspecification
- Sample size is moderate to large

**Types of robust SEs:**
- **HC0-HC4**: Different corrections for leverage and sample size
  - HC3 recommended for general use
  - HC4 best when there are influential points
- **Bootstrap**: Non-parametric approach, good for complex models
  - More computationally intensive
  - Provides full sampling distribution

**Key insight:** Robust SEs protect your inference (p-values, CIs) but don't change the coefficient estimates. If estimates are badly biased by outliers, consider robust regression methods instead (e.g., Huber regression, quantile regression).

## Quick Reference: Using Aurora-GLM's Robust Inference

All the robust inference methods demonstrated in this notebook are now available directly from Aurora-GLM:

```python
from aurora.inference import robust_covariance, bootstrap_inference

# HC standard errors (5 variants available)
robust_result = robust_covariance(result, hc_type='HC3')

# Bootstrap inference
boot_result = bootstrap_inference(result, n_bootstrap=1000, seed=42)
```

**Available HC types:**
- `'HC0'`: White (1980) - basic
- `'HC1'`: DF correction  
- `'HC2'`: Leverage correction
- `'HC3'`: Recommended for general use ⭐
- `'HC4'`: Best for influential points

**When to use:**
- Use **HC3** for most cases (good balance)
- Use **bootstrap** for small samples or non-normal errors
- Use **both** as a robustness check

For more details, see: `aurora/inference/robust_README.md`